# Karachi Spatial Crime Intelligence — Training & Evaluation (Execution Ready)

This notebook trains an **industry-ready** spatial crime intelligence pipeline:

## What it does
1. Loads `karachi_crime_dataset_cleaned.csv` (from preprocessing)
2. Builds **spatial clusters** using:
   - KMeans (baseline)
   - DBSCAN (density hotspots)
   - HDBSCAN (best, if installed) — otherwise OPTICS fallback
3. Chooses a **final cluster column**
4. Infers **Likely Town** and **Likely Subdivision** per cluster (dominant + probabilities)
5. Builds **total-crime trend projection per cluster** using Linear Regression over monthly counts
6. Provides a **client-style inference function**:
   - Input: latitude, longitude, date
   - Output: likely town, subdivision, cluster id, top town/subdivision probabilities, and next-month projection

## Required input files (in same folder / root)
- `karachi_crime_dataset_cleaned.csv`

## Outputs (saved in same folder / root)
- `spatial_intelligence_artifacts.joblib`
- `cluster_profiles.csv`
- `cluster_monthly_trends.csv`


In [2]:
# 0) Imports
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, OPTICS
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression

import joblib

RANDOM_STATE = 42
DATA_DIR = Path(".")
pd.set_option("display.max_rows", 200)


In [3]:
# 1) Load cleaned dataset
df = pd.read_csv(DATA_DIR / "karachi_crime_dataset_cleaned.csv")
print("Loaded:", df.shape)
print("Columns:", list(df.columns))

required = ["lat", "lon", "CRIME_TYPE_NORM"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Re-run preprocessing notebook.")

df.head()


Loaded: (100000, 29)
Columns: ['TOWN', 'TOWN_RISK_LEVEL', 'TOWN_PRIORITY_RANK', 'SUBDIVISION', 'SUBDIVISION_RISK_LEVEL', 'SUBDIVISION_PRIORITY_RANK', 'SEVERITY_SCORE', 'SEVERITY', 'DATE', 'LATITUDE', 'LONGITUDE', 'CRIME_TYPE', 'IS_RED_ZONE', 'IS_ORANGE_ZONE', 'IS_YELLOW_ZONE', 'IS_GREEN_ZONE', 'IS_WHITE_ZONE', 'RISK_ZONE', 'SOURCE', 'RANK', 'CRIME_TYPE_ORIG', 'CRIME_TYPE_NORM', 'lat', 'lon', 'datetime', 'hour', 'day_of_week', 'is_weekend', 'month']


,TOWN,TOWN_RISK_LEVEL,TOWN_PRIORITY_RANK,SUBDIVISION,SUBDIVISION_RISK_LEVEL,SUBDIVISION_PRIORITY_RANK,SEVERITY_SCORE,SEVERITY,DATE,LATITUDE,...,RANK,CRIME_TYPE_ORIG,CRIME_TYPE_NORM,lat,lon,datetime,hour,day_of_week,is_weekend,month
0,Lyari Town,High,1,Baghdadi,High,1,10,High,15/12/2024,24.8592,...,1,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,15/12/2024,0,6,1,12
1,Lyari Town,High,1,Baghdadi,High,1,10,High,03/12/2020,24.8592,...,1,Murder,MURDER,24.8592,66.9998,03/12/2020,0,3,0,12
2,Lyari Town,High,1,Baghdadi,High,1,10,High,24/01/2025,24.8592,...,1,Murder,MURDER,24.8592,66.9998,24/01/2025,0,4,0,1
3,Lyari Town,High,1,Baghdadi,High,1,10,High,18/02/2023,24.8592,...,1,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,18/02/2023,0,5,1,2
4,Lyari Town,High,1,Baghdadi,High,1,10,High,13/04/2025,24.8592,...,1,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,13/04/2025,0,6,1,4


## 2) Temporal features (create if missing)

We use only **DATE-derived** temporal features (no leakage):
- `month`, `day_of_week`, `is_weekend`


In [4]:
# 2) Create temporal features if needed
if "DATE" in df.columns and ("month" not in df.columns or "is_weekend" not in df.columns):
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    df["month"] = df["DATE"].dt.month
    df["day_of_week"] = df["DATE"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

print(df[["DATE","month","day_of_week","is_weekend"]].head(3) if "DATE" in df.columns else "No DATE column present.")


         DATE  month  day_of_week  is_weekend
0  15/12/2024     12            6           1
1  03/12/2020     12            3           0
2  24/01/2025      1            4           0


## 3) Spatial sanity filtering (recommended)

In [5]:
# 3) Spatial sanity filtering (Karachi approx bounding box)
before = len(df)
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

df = df.dropna(subset=["lat","lon"]).copy()
df = df[df["lat"].between(24.6, 25.3) & df["lon"].between(66.8, 67.5)].copy()
after = len(df)

print(f"Rows kept after coordinate filtering: {after} (removed {before-after})")
print("Lat range:", (df["lat"].min(), df["lat"].max()))
print("Lon range:", (df["lon"].min(), df["lon"].max()))


Rows kept after coordinate filtering: 99438 (removed 562)
Lat range: (np.float64(24.797538), np.float64(25.07703))
Lon range: (np.float64(66.961173), np.float64(67.366157))


## 4) Build spatial matrix (scaled)

In [6]:
X_spatial = df[["lat","lon"]].values.astype(np.float32)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_spatial)

print("X_scaled:", X_scaled.shape, X_scaled.dtype)


X_scaled: (99438, 2) float32


## 5) KMeans (baseline) — choose best k by silhouette

In [7]:
sil = {}
k_range = range(5, 21)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil[k] = score
    print(f"k={k:2d}  silhouette={score:.4f}")

best_k = max(sil, key=sil.get)
print("\nBest k by silhouette:", best_k, "score:", sil[best_k])

kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df["cluster_kmeans"] = kmeans_final.fit_predict(X_scaled)


k= 5  silhouette=0.5366
k= 6  silhouette=0.5295
k= 7  silhouette=0.5418
k= 8  silhouette=0.5610
k= 9  silhouette=0.5680
k=10  silhouette=0.5583
k=11  silhouette=0.5497
k=12  silhouette=0.5660
k=13  silhouette=0.5648
k=14  silhouette=0.5733
k=15  silhouette=0.5707
k=16  silhouette=0.5623
k=17  silhouette=0.5205
k=18  silhouette=0.5806
k=19  silhouette=0.5218
k=20  silhouette=0.5325

Best k by silhouette: 18 score: 0.5805656909942627


## 6) DBSCAN (hotspot clustering)

In [8]:
DBSCAN_EPS = 0.25
DBSCAN_MIN_SAMPLES = 50

db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
df["cluster_dbscan"] = db.fit_predict(X_scaled)

n_clusters = df.loc[df["cluster_dbscan"] != -1, "cluster_dbscan"].nunique()
n_noise = int((df["cluster_dbscan"] == -1).sum())

print("DBSCAN clusters (excluding noise):", n_clusters)
print("DBSCAN noise points:", n_noise, f"({n_noise/len(df):.2%})")


DBSCAN clusters (excluding noise): 25
DBSCAN noise points: 0 (0.00%)


## 7) HDBSCAN (best) — fallback to OPTICS

In [9]:
final_density_algo = None

try:
    import hdbscan
    HDB_MIN_CLUSTER_SIZE = 200
    HDB_MIN_SAMPLES = 50

    hdb = hdbscan.HDBSCAN(
        min_cluster_size=HDB_MIN_CLUSTER_SIZE,
        min_samples=HDB_MIN_SAMPLES,
        metric="euclidean"
    )
    df["cluster_hdbscan"] = hdb.fit_predict(X_scaled)
    final_density_algo = "HDBSCAN"

    n_clusters = df.loc[df["cluster_hdbscan"] != -1, "cluster_hdbscan"].nunique()
    n_noise = int((df["cluster_hdbscan"] == -1).sum())

    print("HDBSCAN clusters (excluding noise):", n_clusters)
    print("HDBSCAN noise points:", n_noise, f"({n_noise/len(df):.2%})")

except Exception as e:
    print("HDBSCAN not available or failed:", e)
    print("Falling back to OPTICS...")

    optics = OPTICS(min_samples=DBSCAN_MIN_SAMPLES, cluster_method="xi")
    df["cluster_optics"] = optics.fit_predict(X_scaled)
    final_density_algo = "OPTICS"

    n_clusters = df.loc[df["cluster_optics"] != -1, "cluster_optics"].nunique()
    n_noise = int((df["cluster_optics"] == -1).sum())

    print("OPTICS clusters (excluding noise):", n_clusters)
    print("OPTICS noise points:", n_noise, f"({n_noise/len(df):.2%})")


HDBSCAN not available or failed: No module named 'hdbscan'
Falling back to OPTICS...
OPTICS clusters (excluding noise): 160
OPTICS noise points: 0 (0.00%)


C:\Users\123\AppData\Roaming\Python\Python310\site-packages\sklearn\cluster\_optics.py:1084: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


## 8) Choose final clustering column

In [10]:
if "cluster_hdbscan" in df.columns:
    FINAL_CLUSTER_COL = "cluster_hdbscan"
elif "cluster_optics" in df.columns:
    FINAL_CLUSTER_COL = "cluster_optics"
else:
    FINAL_CLUSTER_COL = "cluster_dbscan"

print("✅ FINAL_CLUSTER_COL =", FINAL_CLUSTER_COL)
print("Clusters (excl noise):", df.loc[df[FINAL_CLUSTER_COL] != -1, FINAL_CLUSTER_COL].nunique())
print("Noise points:", int((df[FINAL_CLUSTER_COL] == -1).sum()))


✅ FINAL_CLUSTER_COL = cluster_optics
Clusters (excl noise): 160
Noise points: 0


## 9) Cluster → Town/Subdivision inference (dominant + probability)

In [11]:
if "TOWN" not in df.columns or "SUBDIVISION" not in df.columns:
    raise ValueError("Expected columns TOWN and SUBDIVISION for inference.")

work = df[df[FINAL_CLUSTER_COL] != -1].copy()

def topk_share(series, k=3):
    vc = series.fillna("UNKNOWN").astype(str).value_counts(normalize=True).head(k)
    return [(idx, float(val)) for idx, val in vc.items()]

profiles = []
for cid, g in work.groupby(FINAL_CLUSTER_COL):
    town_top = topk_share(g["TOWN"], k=3)
    sub_top  = topk_share(g["SUBDIVISION"], k=3)
    profiles.append({
        "cluster_id": int(cid),
        "incidents": int(len(g)),
        "dominant_town": town_top[0][0] if town_top else "UNKNOWN",
        "dominant_town_share": town_top[0][1] if town_top else np.nan,
        "top3_towns": town_top,
        "dominant_subdivision": sub_top[0][0] if sub_top else "UNKNOWN",
        "dominant_subdivision_share": sub_top[0][1] if sub_top else np.nan,
        "top3_subdivisions": sub_top,
        "centroid_lat": float(g["lat"].mean()),
        "centroid_lon": float(g["lon"].mean()),
    })

cluster_profiles = pd.DataFrame(profiles).sort_values("incidents", ascending=False).reset_index(drop=True)
display(cluster_profiles.head(10))

cluster_profiles.to_csv(DATA_DIR / "cluster_profiles.csv", index=False)
print("✅ Saved: cluster_profiles.csv")


,cluster_id,incidents,dominant_town,dominant_town_share,top3_towns,dominant_subdivision,dominant_subdivision_share,top3_subdivisions,centroid_lat,centroid_lon
0,69,3356,Gulberg Town,1.0,"[(Gulberg Town, 1.0)]",Federal B Area Block 17,0.083433,"[(Federal B Area Block 17, 0.08343265792610251...",24.937149,67.087613
1,151,2962,Shah Faisal Town,1.0,"[(Shah Faisal Town, 1.0)]",Shah Faisal Colony No 5,0.286968,"[(Shah Faisal Colony No 5, 0.28696826468602293...",24.881096,67.145214
2,90,2413,Orangi Town,1.0,"[(Orangi Town, 1.0)]",Sector 16,0.222959,"[(Sector 16, 0.22295897223373395), (Sector 13,...",24.946106,67.012881
3,0,2143,Lyari Town,1.0,"[(Lyari Town, 1.0)]",Baghdadi,1.000000,"[(Baghdadi, 1.0)]",24.859200,66.999800
4,11,2077,Lyari Town,1.0,"[(Lyari Town, 1.0)]",Kalakot,1.000000,"[(Kalakot, 1.0)]",24.869417,67.003648
5,139,1889,Landhi Town,1.0,"[(Landhi Town, 1.0)]",Landhi No. 1,0.285336,"[(Landhi No. 1, 0.2853361566966649), (Landhi N...",24.852093,67.186472
6,134,1745,Korangi Town,1.0,"[(Korangi Town, 1.0)]",Zaman Town,1.000000,"[(Zaman Town, 1.0)]",24.836298,67.152710
7,156,1468,Gadap Town,1.0,"[(Gadap Town, 1.0)]",Manghopir,0.664169,"[(Manghopir, 0.6641689373297003), (Gulshan-e-M...",25.025278,67.124094
8,2,1392,Lyari Town,1.0,"[(Lyari Town, 1.0)]",Shah Baig Line,1.000000,"[(Shah Baig Line, 1.0)]",24.864664,66.991471
9,130,1383,Korangi Town,1.0,"[(Korangi Town, 1.0)]",Nasir Colony,1.000000,"[(Nasir Colony, 1.0)]",24.821127,67.128066


✅ Saved: cluster_profiles.csv


In [17]:
# ============================================
# 10) Cluster-level Crime Type Distribution (Historical Likelihood)
# ============================================

# Ensure crime type column exists (from preprocessing)
# If your column name is different, replace CRIME_TYPE_NORM below.
if "CRIME_TYPE_NORM" not in df.columns:
    raise ValueError("Expected column CRIME_TYPE_NORM. Re-run preprocessing notebook.")

crime_dist_rows = []

for cid, g in work.groupby(FINAL_CLUSTER_COL):
    # Normalized distribution of crime types in this cluster
    vc = g["CRIME_TYPE_NORM"].fillna("UNKNOWN").astype(str).value_counts(normalize=True)

    top5 = [(c, float(p)) for c, p in vc.head(5).items()]
    bottom5 = [(c, float(p)) for c, p in vc.tail(5).items()]

    crime_dist_rows.append({
        "cluster_id": int(cid),
        "top5_crimes": top5,
        "bottom5_crimes": bottom5,
    })

cluster_crime_dist = (
    pd.DataFrame(crime_dist_rows)
      .sort_values("cluster_id")
      .reset_index(drop=True)
)

display(cluster_crime_dist.head(10))

# Save for inspection
cluster_crime_dist.to_csv(DATA_DIR / "cluster_crime_dist.csv", index=False)
print("✅ Saved: cluster_crime_dist.csv")


,cluster_id,top5_crimes,bottom5_crimes
0,0,"[(GANG_VIOLENCE, 0.2916472235184321), (MURDER,...","[(Industrial Hazards, 0.01726551563229118), (T..."
1,1,"[(MURDER, 0.16741405082212257), (GANG_VIOLENCE...","[(Kidnapping, 0.052316890881913304), (Extortio..."
2,2,"[(GANG_VIOLENCE, 0.15373563218390804), (MURDER...","[(THEFT, 0.05890804597701149), (Industrial Haz..."
3,3,"[(GANG_VIOLENCE, 0.16258351893095768), (MURDER...","[(Drug Possession, 0.053452115812917596), (Pol..."
4,4,"[(Drug Possession, 0.12694877505567928), (MURD...","[(ASSAULT, 0.05790645879732739), (Land Grabbin..."
5,5,"[(SNATCHING, 0.3865276663993585), (THEFT, 0.38...","[(Drug Possession, 0.019246190858059342), (Lan..."
6,6,"[(ROBBERY, 0.38348323793949307), (SNATCHING, 0...","[(Land Grabbing, 0.019623875715453803), (Indus..."
7,7,"[(THEFT, 0.12743628185907047), (ROBBERY, 0.121...","[(Land Grabbing, 0.05997001499250375), (Indust..."
8,8,"[(SNATCHING, 0.40797546012269936), (THEFT, 0.3...","[(Industrial Hazards, 0.015337423312883436), (..."
9,9,"[(GANG_VIOLENCE, 0.381993006993007), (Drug Pos...","[(ASSAULT, 0.02097902097902098), (Political Vi..."


✅ Saved: cluster_crime_dist.csv


## 10) Total crime trend projection per cluster (monthly)

In [20]:

cluster_trends = pd.DataFrame()

if "DATE" not in df.columns:
    print("No DATE column; skipping trend projection.")
else:
    work2 = work.copy()
    work2["DATE"] = pd.to_datetime(work2["DATE"], errors="coerce")

    if work2["DATE"].isna().all():
        print("DATE could not be parsed; skipping trend projection.")
    else:
        work2 = work2.dropna(subset=["DATE"])
        work2["year_month"] = work2["DATE"].dt.to_period("M").astype(str)

        trends = []
        for cid, g in work2.groupby(FINAL_CLUSTER_COL):
            monthly = (
                g.groupby("year_month")
                 .size()
                 .reset_index(name="crime_count")
                 .sort_values("year_month")
            )
            if len(monthly) < 3:
                continue

            monthly["t"] = np.arange(len(monthly), dtype=np.int32)

            X = monthly[["t"]].values
            y = monthly["crime_count"].values

            lr = LinearRegression()
            lr.fit(X, y)

            r2 = float(lr.score(X, y))
            slope = float(lr.coef_[0])
            next_t = int(monthly["t"].max() + 1)
            next_pred = float(lr.predict(np.array([[next_t]]))[0])

            trends.append({
                "cluster_id": int(cid),
                "n_months": int(len(monthly)),
                "slope_per_month": slope,
                "r2": r2,
                "last_month": monthly["year_month"].iloc[-1],
                "last_count": int(monthly["crime_count"].iloc[-1]),
                "next_month_projection": max(0.0, next_pred),
            })

        cluster_trends = (
            pd.DataFrame(trends)
              .sort_values("next_month_projection", ascending=False)
              .reset_index(drop=True)
        )

        display(cluster_trends.head(10))
        cluster_trends.to_csv(DATA_DIR / "cluster_monthly_trends.csv", index=False)
        print("✅ Saved: cluster_monthly_trends.csv")


C:\Users\123\AppData\Local\Temp\ipykernel_18112\1263013666.py:7: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  work2["DATE"] = pd.to_datetime(work2["DATE"], errors="coerce")


,cluster_id,n_months,slope_per_month,r2,last_month,last_count,next_month_projection
0,69,60,0.036955,0.006056,2025-05,54,57.060452
1,151,60,0.020617,0.002464,2025-05,52,49.995480
2,90,60,-0.020200,0.004274,2025-05,34,39.600565
3,11,60,0.075382,0.044885,2025-05,46,36.915819
4,0,60,0.008197,0.000644,2025-05,45,35.966667
5,139,60,0.033871,0.011540,2025-05,41,32.516384
6,134,60,0.054932,0.030090,2025-05,40,30.758757
7,156,60,0.000945,0.000009,2025-05,22,24.495480
8,130,60,0.031926,0.013761,2025-05,27,24.023729
9,2,60,0.006835,0.000632,2025-05,28,23.408475


✅ Saved: cluster_monthly_trends.csv


## 11) Inference: (lat, lon, date) → likely Town/Subdivision + projection

For density-based clusters we assign by **nearest labeled neighbor**.


In [21]:
# ============================================
# 12) Inference: (lat, lon, date) → likely Town/Subdivision + crimes + projection
# ============================================

train_mask = (df[FINAL_CLUSTER_COL] != -1)
X_train_scaled = X_scaled[train_mask.values]
y_cluster = df.loc[train_mask, FINAL_CLUSTER_COL].astype(int).values

nn = NearestNeighbors(n_neighbors=1)
nn.fit(X_train_scaled)

profile_by_cluster = {int(r["cluster_id"]): r for _, r in cluster_profiles.iterrows()}
trend_by_cluster = {int(r["cluster_id"]): r for _, r in cluster_trends.iterrows()} if not cluster_trends.empty else {}

# ✅ NEW: crime distribution lookup
crime_by_cluster = {int(r["cluster_id"]): r for _, r in cluster_crime_dist.iterrows()}

def predict_location_and_trend(lat: float, lon: float, date: str | None = None):
    x = np.array([[lat, lon]], dtype=np.float32)
    x_scaled = scaler.transform(x)

    dist, idx = nn.kneighbors(x_scaled, n_neighbors=1, return_distance=True)
    cid = int(y_cluster[int(idx[0][0])])

    prof = profile_by_cluster.get(cid, None)
    trend = trend_by_cluster.get(cid, None)
    crime_info = crime_by_cluster.get(cid, None)

    out = {
        "cluster_id": cid,
        "nearest_distance": float(dist[0][0]),

        "likely_town": prof["dominant_town"] if prof is not None else None,
        "likely_subdivision": prof["dominant_subdivision"] if prof is not None else None,
        "top3_towns": prof["top3_towns"] if prof is not None else None,
        "top3_subdivisions": prof["top3_subdivisions"] if prof is not None else None,
        "cluster_incidents": int(prof["incidents"]) if prof is not None else None,

        # ✅ NEW: crime likelihoods
        "top_5_most_likely_crimes": crime_info["top5_crimes"] if crime_info is not None else None,
        "top_5_least_likely_crimes": crime_info["bottom5_crimes"] if crime_info is not None else None,

        # Trend outputs (optional)
        "next_month_projection": float(trend["next_month_projection"]) if trend is not None else None,
        "trend_slope_per_month": float(trend["slope_per_month"]) if trend is not None else None,
        "trend_r2": float(trend["r2"]) if trend is not None else None,
        "last_month": trend["last_month"] if trend is not None else None,
        "last_count": int(trend["last_count"]) if trend is not None else None,
    }

    if date:
        dt = pd.to_datetime(date, errors="coerce")
        out["input_date"] = str(dt.date()) if pd.notna(dt) else None
        out["input_is_weekend"] = int(dt.dayofweek in [5, 6]) if pd.notna(dt) else None

    return out

# Example client query
predict_location_and_trend(24.8607, 67.0011, "2026-01-04")


{'cluster_id': 0,
 'nearest_distance': 0.03260077439498229,
 'likely_town': 'Lyari Town',
 'likely_subdivision': 'Baghdadi',
 'top3_towns': [('Lyari Town', 1.0)],
 'top3_subdivisions': [('Baghdadi', 1.0)],
 'cluster_incidents': 2143,
 'top_5_most_likely_crimes': [('GANG_VIOLENCE', 0.2916472235184321),
  ('MURDER', 0.2813812412505833),
  ('GUN_VIOLENCE', 0.21885207652823147),
  ('ROBBERY', 0.036397573495100324),
  ('SNATCHING', 0.032197853476434904)],
 'top_5_least_likely_crimes': [('Industrial Hazards', 0.01726551563229118),
  ('THEFT', 0.015865608959402706),
  ('Political Violence', 0.014932337844143724),
  ('Extortion', 0.013065795613625758),
  ('Drug Possession', 0.011199253383107793)],
 'next_month_projection': 35.96666666666667,
 'trend_slope_per_month': 0.0081967213114754,
 'trend_r2': 0.0006444020538732831,
 'last_month': '2025-05',
 'last_count': 45,
 'input_date': '2026-01-04',
 'input_is_weekend': 1}

## 12) Save deployment artifacts

In [22]:
# ============================================
# 13) Save deployment artifacts
# ============================================

artifacts = {
    "FINAL_CLUSTER_COL": FINAL_CLUSTER_COL,
    "scaler": scaler,
    "kmeans_final": kmeans_final,
    "dbscan": db,
    "density_algo_name": final_density_algo,
    "nearest_neighbors": nn,
    "X_train_scaled": X_train_scaled,
    "y_cluster": y_cluster,
    "cluster_profiles": cluster_profiles,
    "cluster_trends": cluster_trends,

    # ✅ NEW: save crime distribution
    "cluster_crime_dist": cluster_crime_dist,
}

if "hdb" in locals():
    artifacts["hdbscan_model"] = hdb
if "optics" in locals():
    artifacts["optics_model"] = optics

joblib.dump(artifacts, DATA_DIR / "spatial_intelligence_artifacts.joblib")
print("✅ Saved: spatial_intelligence_artifacts.joblib")


✅ Saved: spatial_intelligence_artifacts.joblib


In [24]:
import joblib
import numpy as np
import pandas as pd

# 1) Load artifacts
artifacts = joblib.load("spatial_intelligence_artifacts.joblib")

scaler = artifacts["scaler"]
nn = artifacts["nearest_neighbors"]               # already fitted NearestNeighbors
y_cluster = artifacts["y_cluster"]                # cluster label for each NN reference point

cluster_profiles = artifacts["cluster_profiles"]
cluster_trends = artifacts.get("cluster_trends", pd.DataFrame())

# ✅ NEW: cluster-level crime distribution
cluster_crime_dist = artifacts.get("cluster_crime_dist", pd.DataFrame())

# Build lookup dicts
profile_by_cluster = {int(r["cluster_id"]): r for _, r in cluster_profiles.iterrows()}

trend_by_cluster = (
    {int(r["cluster_id"]): r for _, r in cluster_trends.iterrows()}
    if not cluster_trends.empty else {}
)

crime_by_cluster = (
    {int(r["cluster_id"]): r for _, r in cluster_crime_dist.iterrows()}
    if not cluster_crime_dist.empty else {}
)

# 2) Define prediction function
def predict_location_and_trend(lat, lon, date=None):
    # Scale input
    x = np.array([[lat, lon]], dtype=np.float32)
    x_scaled = scaler.transform(x)

    # Nearest neighbor → cluster
    dist, idx = nn.kneighbors(x_scaled, n_neighbors=1, return_distance=True)
    cid = int(y_cluster[int(idx[0][0])])

    prof = profile_by_cluster.get(cid)
    trend = trend_by_cluster.get(cid)
    crime_info = crime_by_cluster.get(cid)

    result = {
        "cluster_id": cid,
        "nearest_distance": float(dist[0][0]),

        "likely_town": prof["dominant_town"] if prof is not None else None,
        "likely_subdivision": prof["dominant_subdivision"] if prof is not None else None,
        "top_3_towns": prof["top3_towns"] if prof is not None else None,
        "top_3_subdivisions": prof["top3_subdivisions"] if prof is not None else None,
        "total_incidents_in_cluster": int(prof["incidents"]) if prof is not None else None,

        # ✅ NEW: crime likelihoods from historical distribution
        "top_5_most_likely_crimes": crime_info["top5_crimes"] if crime_info is not None else None,
        "top_5_least_likely_crimes": crime_info["bottom5_crimes"] if crime_info is not None else None,
    }

    if trend is not None:
        result.update({
            "crime_trend_slope_per_month": round(float(trend["slope_per_month"]), 2),
            "trend_r2": round(float(trend["r2"]), 2),
            "last_month": trend["last_month"],
            "last_month_crime_count": int(trend["last_count"]),
            "next_month_projection": int(round(float(trend["next_month_projection"]))),
        })

    if date:
        dt = pd.to_datetime(date, errors="coerce")
        result["input_date"] = str(dt.date()) if pd.notna(dt) else None
        result["is_weekend"] = int(dt.dayofweek in [5, 6]) if pd.notna(dt) else None

    return result


# 3) Ask for user input
lat = float(input("Enter latitude: "))
lon = float(input("Enter longitude: "))
date = input("Enter date (YYYY-MM-DD) or press Enter to skip: ").strip()
date = date if date else None

# 4) Run prediction
output = predict_location_and_trend(lat, lon, date)

# 5) Pretty print result
print("\n===== LOCATION CRIME INTELLIGENCE RESULT =====")
for k, v in output.items():
    print(f"{k}: {v}")


Enter latitude:  24.928450232548883
Enter longitude:  67.04808096011222
Enter date (YYYY-MM-DD) or press Enter to skip:  2026-01-05



===== LOCATION CRIME INTELLIGENCE RESULT =====
cluster_id: 81
nearest_distance: 0.0832379671069044
likely_town: North Nazimabad Town
likely_subdivision: Block G
top_3_towns: [('North Nazimabad Town', 1.0)]
top_3_subdivisions: [('Block G', 1.0)]
total_incidents_in_cluster: 214
top_5_most_likely_crimes: [('GANG_VIOLENCE', 0.14953271028037382), ('MURDER', 0.11214953271028037), ('Drug Possession', 0.102803738317757), ('SNATCHING', 0.09345794392523364), ('Land Grabbing', 0.08411214953271028)]
top_5_least_likely_crimes: [('THEFT', 0.07476635514018691), ('ASSAULT', 0.07009345794392523), ('Industrial Hazards', 0.056074766355140186), ('Political Violence', 0.0514018691588785), ('Extortion', 0.0514018691588785)]
crime_trend_slope_per_month: -0.01
trend_r2: 0.02
last_month: 2025-05
last_month_crime_count: 3
next_month_projection: 3
input_date: 2026-01-05
is_weekend: 0
